# Revision 2 — teacher-removal study

Reviewer 1 comment 2: does removing the lowest-original-raw-Shapley teacher outperform uniformly random teacher removal?

Use all ten original seeds on both UCI-HAR and corrected WISDM. For every bank, train all five four-teacher students with uniform retained weights, plus a matched five-teacher uniform control. All six conditions share the original final-student seed (`outer*1000000+7`), split, architecture, optimizer, temperature, 60-epoch limit and validation checkpoint/patience rule.

**120 research student fits: 100 removals + 20 controls. No teacher retraining.** Each bank's removal choice is frozen from its original raw validation Shapley values. Exact ties use the earliest saved teacher index. The mean of all five removal scores is the exact expected score of uniform random removal, including the attribution-selected outcome. Do not pick the best removal using new validation or test outcomes.

All 100 saved teachers must reproduce original validation accuracy and macro-F1 within 0.0001 before smoke or research can start. Deterministic operations and full float32 precision are enabled before teacher inference. This preflight caches teacher targets; later fits reuse them. Source hashes, split, targets and environment enter saved identities. Existing production results remain read only.

All six predeclared conditions are evaluated on the already-used fixed test split with validation-only checkpoint selection. This is post-review robustness evidence. The analysis uses ten seed-paired differences per dataset; five removals are not five independent replicates. Six prespecified paired comparisons across both datasets share one Holm correction family.

Run cells in order with a GPU. The separate twelve-fit smoke experiment checks all conditions at two epochs. Research saves every fit, model and prediction-label record under `MyDrive/makd-har/results/revision-2/teacher-pruning/`. Rerunning the research cell resumes compatible records; changed provenance stops reuse.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT=Path('/content/drive/MyDrive/makd-har')
EXPERIMENT=PROJECT/'revision_2/teacher_pruning'
assert (EXPERIMENT/'frozen-inputs.json').is_file(), 'Wait for Drive synchronization.'
print('Project:',PROJECT)


In [ ]:
# Preserve the installed Colab TensorFlow/Keras environment.
import subprocess,sys,json,os,datetime
import tensorflow as tf
print('TensorFlow',tf.__version__,'Keras',tf.keras.__version__)
print('GPU',tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):raise RuntimeError('Choose a GPU runtime before continuing.')

def run_pruning(*flags):
    label='verify' if '--verify-only' in flags else ('preflight' if '--preflight-only' in flags else ('smoke' if '--smoke' in flags else 'research'))
    logs=PROJECT/'results/revision-2/diagnostics';logs.mkdir(parents=True,exist_ok=True)
    log=logs/f"teacher-pruning-{label}-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}.log"
    command=[sys.executable,'-u',str(EXPERIMENT/'run_teacher_pruning.py'),'--drive-root',str(PROJECT),*flags]
    print('Full diagnostic log:',log,flush=True)
    notices=0
    with log.open('w') as stream:
        with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,
                              env=dict(os.environ,TF_CPP_MIN_LOG_LEVEL='2')) as process:
            for line in process.stdout:
                stream.write(line);stream.flush()
                if 'NodeDef mentions attribute use_unbounded_threadpool' in line:
                    notices+=1
                else:print(line,end='',flush=True)
            code=process.wait()
    if notices:print(f'{notices} repeated TensorFlow compatibility notices retained in the full log.')
    if code:raise RuntimeError(f'{label} stopped (exit {code}). Full error saved in {log}.')


In [ ]:
# No training: verify frozen files, then all twenty banks and one hundred saved teachers.
run_pruning('--verify-only')
run_pruning('--preflight-only')
gate=json.loads((PROJECT/'results/revision-2/teacher-pruning/preflight-completed.json').read_text())
assert gate['passed'] and gate['teacher_checks']==100 and gate['banks']==20
print('ALL PREFLIGHT CHECKS PASSED.')


In [ ]:
# Separate smoke experiment: six conditions per dataset, two epochs each.
run_pruning('--smoke')
smoke=json.loads((PROJECT/'results/revision-2/smoke-teacher-pruning/completed.json').read_text())
assert smoke['status']=='complete' and smoke['mode']=='smoke' and smoke['fits']==12
print('SMOKE PASSED: all twelve short fits completed.')


In [ ]:
# Research: one hundred removal fits plus twenty matched full-teacher controls.
smoke=json.loads((PROJECT/'results/revision-2/smoke-teacher-pruning/completed.json').read_text())
assert smoke['status']=='complete' and smoke['mode']=='smoke' and smoke['fits']==12
run_pruning()


In [ ]:
completed=PROJECT/'results/revision-2/teacher-pruning/completed.json'
assert completed.is_file(), 'Research incomplete. Rerun only the research cell to resume verified fits.'
record=json.loads(completed.read_text())
assert record['status']=='complete' and record['mode']=='research' and record['fits']==120
print('COMPLETE: 120 research fits. Ready for OneDrive verification and analysis.')


## After completion

Keep all outcomes, including null or negative results. Compare the frozen lowest-value removal with expected random removal and the matched full-teacher student. Record the original UCI 20-epoch surrogate attribution and corrected WISDM 60-epoch attribution explicitly. This experiment tests those original selection rules; it does not produce confidence intervals for every original Shapley value.

There is no test-based choice of removal, seed or weighting rule. Later analysis verifies hashes, saved predictions, metrics and paired comparisons before updating the manuscript. The original ten-seed main table stays unchanged.
